In [ ]:
# ── Starter Cell 1: Load environment variables ──────────────────────────────
from dotenv import load_dotenv   # imports the dotenv loader function
import os                        # gives us os.getenv() to read env vars

load_dotenv()                    # reads the .env file in the working directory and injects vars into os.environ

api_key = os.getenv("ANTHROPIC_API_KEY")   # retrieves the key; returns None if not found

print("API key loaded:", api_key is not None)  # True = key was found; False = check your .env file

In [ ]:
# ── Starter Cell 2: Initialise the Anthropic client ─────────────────────────
from anthropic import Anthropic   # imports the main SDK class

client = Anthropic(api_key=api_key)  # creates one reusable client; passing api_key explicitly is safer than relying on the env var being auto-detected

print("Anthropic client ready")     # confirms the object was constructed without errors

In [ ]:
# ── Starter Cell 3: Smoke test — confirms end-to-end connectivity ────────────
response = client.messages.create(
    model="claude-haiku-4-5",        # cheapest/fastest model — ideal for smoke tests
    #    model="claude-sonnet-4-5",  # uncomment to test with a larger model
    max_tokens=50,                   # hard upper bound on output tokens; prevents runaway generation
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]  # minimal valid message list
)

print(response.content[0].text)     # response.content is a list of content blocks; [0] is the first (and here only) block; .text extracts the string

# CCA Lab: Being Clear and Direct

**Course:** Anthropic Academy — Prompt Engineering Fundamentals  
**Sub-module:** Crafting Effective Prompts  
**Lesson:** Being Clear and Direct

---

## What this lab covers
- Why vague prompts produce inconsistent, off-target responses
- How to specify **format**, **constraints**, **audience**, and **context** explicitly
- The `system` parameter: what it is, when to use it, and why it matters
- Multi-turn conversations: accumulating the `messages` list correctly
- An improvement cycle: write → evaluate → improve → re-evaluate
- Failure mode analysis: what breaks when prompts are under-specified
- Token usage tracking across an entire session

## CCA Domains Exercised
- Prompt Construction · Output Control · System Prompts · Evaluation & Iteration

## Learning Objectives
1. Identify the four dimensions of a clear prompt: format, constraint, audience, context
2. Use the `system` parameter to set persistent instructions vs. per-turn user content
3. Build and extend a multi-turn `messages` list manually
4. Execute and interpret a write → evaluate → improve → re-evaluate cycle
5. Diagnose and fix the top failure modes caused by vague prompts
6. Track and interpret token usage across a full session

## Section 1: Prerequisites and Environment Setup
## CCA objective demonstrated: Establish a reproducible, observable API environment

Before writing any prompt, confirm the runtime is healthy. The three starter cells above cover:
- **dotenv load** — reads `ANTHROPIC_API_KEY` from `.env`
- **client init** — constructs the `Anthropic` object once and reuses it everywhere
- **smoke test** — a live round-trip to the API proving connectivity

### Required packages
```
pip install anthropic python-dotenv
```

### `.env` file (same directory as this notebook)
```
ANTHROPIC_API_KEY=sk-ant-...
```

We also initialise a **global usage log** here — every API call in this lab appends one entry so we can analyse token spend at the end.

In [ ]:
# ── Section 1 setup: global usage log and shared model constant ──────────────

usage_log = []          # list of dicts; one entry per API call; grows throughout the notebook

MODEL = "claude-haiku-4-5"   # single source of truth for model name; change here to affect all cells

def log_usage(label, response):
    """Append one token-usage record to usage_log.

    Parameters
    ----------
    label    : str   — human-readable name for the call (e.g. 'smoke-test')
    response : anthropic.types.Message — the raw API response object

    Returns
    -------
    None  (side-effect: appends to usage_log)
    """
    entry = {
        "label":         label,                        # identifies which call this was
        "input_tokens":  response.usage.input_tokens,  # tokens consumed by the prompt + history
        "output_tokens": response.usage.output_tokens, # tokens generated in the reply
        "stop_reason":   response.stop_reason,         # 'end_turn' = natural finish; 'max_tokens' = truncated
        "truncated":     response.stop_reason == "max_tokens",  # True flags a potentially cut-off response
    }
    usage_log.append(entry)   # mutates the module-level list so every cell can see it

print("usage_log initialised:", usage_log)   # should print an empty list []

## Section 2: API Glossary and Reference
## CCA objective demonstrated: Understand every parameter before using it

| Parameter | Type | Required | What it does |
|-----------|------|----------|--------------|
| `model` | `str` | ✅ | Which Claude model to call (e.g. `claude-haiku-4-5`) |
| `max_tokens` | `int` | ✅ | Hard ceiling on output tokens; response is truncated if hit |
| `messages` | `list[dict]` | ✅ | Alternating `user`/`assistant` turns; the full conversation history |
| `system` | `str` | ❌ | Persistent instructions evaluated before any user turn |
| `temperature` | `float 0–1` | ❌ | Randomness of sampling; 0 = deterministic, 1 = creative |
| `stop_sequences` | `list[str]` | ❌ | Model stops generating when it produces one of these strings |
| `response.content` | `list` | — | List of content blocks in the reply (usually length 1) |
| `response.content[0].text` | `str` | — | The plain-text string of the first content block |
| `response.usage.input_tokens` | `int` | — | Tokens charged for the prompt + history |
| `response.usage.output_tokens` | `int` | — | Tokens charged for the generated reply |
| `response.stop_reason` | `str` | — | `'end_turn'` = finished naturally; `'max_tokens'` = truncated |

> **Why is `max_tokens` required?** Claude has no default limit — omitting it risks very long (and expensive) responses. Always set it explicitly.

## Section 3: Client Setup and the `ask()` Helper
## CCA objective demonstrated: Build a reusable, annotated API wrapper

We wrap `client.messages.create()` in a thin helper so every call automatically:
- logs tokens to `usage_log`
- surfaces `stop_reason` for truncation detection
- accepts an optional `system` prompt

Every statement is annotated below.

In [ ]:
# ── Section 3: The ask() helper with statement-by-statement annotation ───────

def ask(prompt, system=None, max_tokens=256, label="call"):
    """Send a single-turn prompt to Claude and return the text reply.

    Parameters
    ----------
    prompt     : str  — the user-turn content
    system     : str  — optional persistent instructions (default None = omitted)
    max_tokens : int  — output ceiling (default 256 is enough for short answers)
    label      : str  — tag written to usage_log for later analysis

    Returns
    -------
    str — the plain-text reply from Claude
    """

    # Build the keyword-argument dict for the API call.
    # 'system' is a top-level keyword argument (not inside 'messages') because
    # the Anthropic API separates system instructions from the conversation turns.
    kwargs = {
        "model":      MODEL,                                      # use the module-level constant
        "max_tokens": max_tokens,                                 # always required; prevents runaway output
        "messages":   [{"role": "user", "content": prompt}],      # minimal single-turn message list
    }

    if system is not None:          # only add 'system' when the caller explicitly provides it
        kwargs["system"] = system   # passed as a keyword arg, NOT as a message dict with role='system'

    response = client.messages.create(**kwargs)   # ** unpacks the dict into named arguments; makes the call

    # response.content is a LIST of content blocks because Claude can return
    # multiple blocks (e.g. text + tool_use). We access [0] for the first block.
    text = response.content[0].text   # .text extracts the string from the TextBlock object

    # stop_reason tells us WHY generation stopped:
    #   'end_turn'   → Claude finished naturally (good)
    #   'max_tokens' → we hit the ceiling; reply may be cut off (investigate!)
    if response.stop_reason == "max_tokens":
        print(f"⚠️  [{label}] stop_reason=max_tokens — consider raising max_tokens")

    log_usage(label, response)   # always record token spend before returning

    return text   # return only the string so callers don't need to unwrap the response object


# Quick sanity check
reply = ask("What is 2 + 2? Answer in one word.", label="sanity-check")
print("Reply:", reply)
print("Usage log so far:", usage_log)

## Section 4: The `system` Parameter — What, When, and Why
## CCA objective demonstrated: Use `system` to set persistent application-layer instructions

### What is `system`?
The `system` parameter is a string of instructions that Claude reads **before** any user turn. It is never shown to the end-user and persists across every turn in a conversation.

> **Architectural principle:** The system prompt is evaluated before any user turn and is invisible to the user; treat it as the **application layer**, not the conversation layer. Configure the model's role, format rules, and guardrails here — not in the first user message.

### What belongs where?

| Instruction type | Put in `system`? | Put in user turn? | Reason |
|---|---|---|---|
| Persistent role / persona | ✅ Yes | ❌ Redundant | Applies to every turn; set once |
| Format rules (apply to all turns) | ✅ Yes | ❌ Redundant | Avoids repeating in every message |
| Guardrails / safety constraints | ✅ Yes | ❌ Redundant | Should not be user-overridable |
| Task-specific context | ❌ No | ✅ Yes | Changes per request |
| One-off output constraints | ❌ No | ✅ Yes | Specific to that single turn |
| The actual user request | ❌ No | ✅ Yes | That's what the user turn is for |

### Components of a well-structured system prompt
1. **Role** — who/what Claude is in this application
2. **Audience** — who it is talking to
3. **Format** — output structure rules that apply globally
4. **Constraint** — hard rules (length, tone, topic scope)

In [ ]:
# ── Section 4: Live demo — with-system vs without-system ────────────────────

# A system prompt with all four components annotated inline
SYSTEM_PROMPT = (
    "You are a friendly Python tutor."              # ROLE: tells Claude its persona for every turn
    " Your audience is beginners with no CS background."  # AUDIENCE: calibrates vocabulary and depth
    " Always respond in plain bullet points."       # FORMAT: applies to every reply — set once here
    " Keep answers under 80 words."                 # CONSTRAINT: hard length rule
)

QUESTION = "What is a variable in Python?"

# --- Call WITHOUT system prompt ---
reply_no_system = ask(
    QUESTION,
    label="no-system",
    max_tokens=300
)  # no system= argument → Claude picks its own format and depth

# --- Call WITH system prompt ---
reply_with_system = ask(
    QUESTION,
    system=SYSTEM_PROMPT,   # system= is a keyword arg separate from messages
    label="with-system",
    max_tokens=300
)

print("=" * 60)
print("WITHOUT system prompt:")
print(reply_no_system)
print()
print("=" * 60)
print("WITH system prompt:")
print(reply_with_system)
print()
print("Usage log entries so far:", len(usage_log))

## Section 5: Multi-Turn Conversation — Messages List Accumulation
## CCA objective demonstrated: Build and extend the `messages` list across turns using `client.messages.create()` directly

The Anthropic API is **stateless** — the server remembers nothing between calls. To create a conversation, the client must send the **entire history** on every turn.

The pattern:
```python
conversation = []
conversation.append({"role": "user", "content": "..."})
resp = client.messages.create(model=..., messages=conversation, ...)
conversation.append({"role": "assistant", "content": resp.content[0].text})
```

Each new turn appends **two** entries: one user message and one assistant reply.

> **First-line clarity principle:** Because the entire history is re-transmitted each turn, the first user message should be maximally clear — it anchors all subsequent turns and is re-read by the model on every API call.

In [ ]:
# ── Section 5: Multi-turn conversation — raw pattern with direct client.messages.create() ──
# We deliberately do NOT use ask() here so the mechanics are fully visible.

conversation = []   # start with an empty history; we will grow this list turn by turn

# ── Turn 1 ──────────────────────────────────────────────────────────────────
conversation.append({"role": "user", "content": "I want to learn about Python lists. What are they?"})
# append adds a user dict to the list; role must be 'user' for the first message

resp1 = client.messages.create(
    model=MODEL,             # use the module-level model constant
    max_tokens=150,          # short reply for this demo
    messages=conversation    # send the full history (just 1 message so far)
)  # returns a Message object with .content, .usage, .stop_reason

conversation.append({"role": "assistant", "content": resp1.content[0].text})
# we store the assistant reply back into the list so the next turn includes it

log_usage("multiturn-turn1", resp1)   # track tokens for turn 1
print("Turn 1 — user: I want to learn about Python lists. What are they?")
print("Turn 1 — assistant:", resp1.content[0].text[:200], "...")
print(f"  input_tokens={resp1.usage.input_tokens}  output_tokens={resp1.usage.output_tokens}")
print()

# ── Turn 2 ──────────────────────────────────────────────────────────────────
conversation.append({"role": "user", "content": "How do I add items to a list?"})
# this message is appended AFTER the assistant reply so the order is user→assistant→user

resp2 = client.messages.create(
    model=MODEL,
    max_tokens=150,
    messages=conversation    # now contains 3 messages: user + assistant + user
)

conversation.append({"role": "assistant", "content": resp2.content[0].text})
log_usage("multiturn-turn2", resp2)
print("Turn 2 — user: How do I add items to a list?")
print("Turn 2 — assistant:", resp2.content[0].text[:200], "...")
print(f"  input_tokens={resp2.usage.input_tokens}  output_tokens={resp2.usage.output_tokens}")
print()

# ── Turn 3 ──────────────────────────────────────────────────────────────────
conversation.append({"role": "user", "content": "Can you show me a short code example using both?"})

resp3 = client.messages.create(
    model=MODEL,
    max_tokens=150,
    messages=conversation    # now contains 5 messages; the full history is re-sent every call
)

conversation.append({"role": "assistant", "content": resp3.content[0].text})
log_usage("multiturn-turn3", resp3)
print("Turn 3 — user: Can you show me a short code example using both?")
print("Turn 3 — assistant:", resp3.content[0].text[:200], "...")
print(f"  input_tokens={resp3.usage.input_tokens}  output_tokens={resp3.usage.output_tokens}")
print()

# ── Token accumulation analysis ──────────────────────────────────────────────
print("=" * 60)
print("Token accumulation across turns (full history re-sent each call):")
print(f"{'Turn':<8} {'input_tokens':<16} {'output_tokens':<16}")
print("-" * 40)
for i, resp in enumerate([resp1, resp2, resp3], start=1):
    print(f"{i:<8} {resp.usage.input_tokens:<16} {resp.usage.output_tokens:<16}")
    # input_tokens grows each turn because the entire conversation history is re-sent

print()
print("▶ Notice: input_tokens grows each turn because the FULL conversation")
print("  history is re-sent to the API on every call — this is the cost of stateless context.")
print(f"  History grew from {len([conversation[0]])} message(s) to {len(conversation)} messages over 3 turns.")

## Section 6: The Four Dimensions of a Clear Prompt
## CCA objective demonstrated: Operationalise clarity into measurable prompt components

A vague prompt leaves four dimensions unspecified. Filling them in is the core skill of this lesson.

| Dimension | Vague | Clear |
|-----------|-------|-------|
| **Format** | (none specified) | "Respond as a numbered list with exactly 5 items" |
| **Constraint** | (none specified) | "Each item must be under 15 words" |
| **Audience** | (none specified) | "Explain for a non-technical manager" |
| **Context** | (none specified) | "The product is a SaaS analytics dashboard" |

In the improvement cycle below we will score prompts against these four dimensions.

In [ ]:
# ── Section 6: Vague vs. clear prompt — side-by-side comparison ─────────────
import re   # needed for pattern-based scoring later

VAGUE_PROMPT = "Tell me about machine learning."
# No format, no constraint, no audience, no context — Claude must guess everything

CLEAR_PROMPT = (
    "You are explaining to a non-technical marketing manager."   # AUDIENCE
    " List exactly 3 practical business benefits of machine learning."  # FORMAT + CONSTRAINT (count)
    " Each benefit must be one sentence, under 20 words."        # CONSTRAINT (length)
    " Focus on e-commerce applications."                          # CONTEXT
)

reply_vague = ask(VAGUE_PROMPT, label="clarity-vague", max_tokens=300)
reply_clear = ask(CLEAR_PROMPT, label="clarity-clear", max_tokens=300)

print("VAGUE prompt reply:")
print(reply_vague[:400], "...")
print()
print("CLEAR prompt reply:")
print(reply_clear)
print()
print(f"Word counts — vague: {len(reply_vague.split())}  clear: {len(reply_clear.split())}")
# The clear reply should be significantly shorter and more structured

## Section 7: Improvement Cycle — Write → Evaluate → Improve → Re-evaluate
## CCA objective demonstrated: Apply a numeric rubric to iterate prompt quality systematically

### Rubric (total = 12 points; pass threshold = 10/12 ≥ 83%)

| Criterion | Points | What earns full marks |
|-----------|--------|----------------------|
| `format` | 0–3 | Reply is a numbered list with exactly 3 items |
| `constraint` | 0–3 | Every item ≤ 20 words |
| `audience` | 0–3 | No jargon; suitable for a non-technical reader |
| `context` | 0–3 | All items relate to e-commerce, not generic ML |

> **Grader reliability note:** Deterministic rubrics can over-score responses that
> contain the right words but wrong semantics. For production evals, complement
> keyword checks with a Claude-as-judge semantic pass — the rule + judge layering
> pattern from the evaluation labs.

In [ ]:
# ── Section 7: Rubric scorer ─────────────────────────────────────────────────

JARGON_WORDS = ["gradient", "backpropagation", "hyperparameter", "tensor", "epoch",
                "neural network", "stochastic", "optimizer", "loss function"]
# These are terms a non-technical manager is unlikely to know

ECOMMERCE_WORDS = ["customer", "product", "purchase", "sales", "revenue", "cart",
                   "recommend", "shop", "inventory", "price", "order", "store"]
# Presence of any of these signals e-commerce context was respected

def evaluate_response(text, label="response"):
    """Score a response against the four-criterion rubric.

    Parameters
    ----------
    text  : str — the model reply to evaluate
    label : str — name used in printed output

    Returns
    -------
    dict with keys 'scores' (dict of criterion→int) and 'total' (int)
    """
    scores = {}

    # FORMAT: count numbered list items using regex
    # re.findall not re.match because items can appear anywhere in the text, not just the start
    # \b ensures we match whole numbers (not '1' inside '10')
    items = re.findall(r'^\s*\d+[.)]\.?\s+\S', text, re.MULTILINE)
    # re.MULTILINE makes ^ match the start of each line, not just the start of the string
    n_items = len(items)   # integer count of numbered list lines found
    if n_items == 3:
        scores["format"] = 3      # exactly 3 items → full marks
    elif n_items in (2, 4):
        scores["format"] = 1      # close but wrong count → partial credit
    else:
        scores["format"] = 0      # missing list structure or wildly wrong count

    # CONSTRAINT: check every sentence/line is ≤ 20 words
    # We split on newlines to approximate item boundaries
    lines = [l.strip() for l in text.splitlines() if l.strip()]   # remove blank lines
    long_lines = [l for l in lines if len(l.split()) > 20]         # list of lines that exceed 20 words
    if len(long_lines) == 0:
        scores["constraint"] = 3   # no lines exceeded the limit
    elif len(long_lines) == 1:
        scores["constraint"] = 1   # one minor violation
    else:
        scores["constraint"] = 0   # multiple violations

    # AUDIENCE: penalise jargon presence
    # re.search searches anywhere in the string (vs re.match which only checks the start)
    # \b word boundaries prevent partial matches like 'epoch' inside 'epochal'
    text_lower = text.lower()   # case-fold once; reuse for all checks
    jargon_found = [w for w in JARGON_WORDS if re.search(r'\b' + re.escape(w) + r'\b', text_lower)]
    if len(jargon_found) == 0:
        scores["audience"] = 3    # no jargon detected
    elif len(jargon_found) == 1:
        scores["audience"] = 1    # one jargon slip
    else:
        scores["audience"] = 0    # too much jargon for a non-technical reader

    # CONTEXT: require at least one e-commerce word
    ecom_found = [w for w in ECOMMERCE_WORDS if re.search(r'\b' + re.escape(w) + r'\b', text_lower)]
    if len(ecom_found) >= 2:
        scores["context"] = 3     # clearly e-commerce focused
    elif len(ecom_found) == 1:
        scores["context"] = 1     # barely on-topic
    else:
        scores["context"] = 0     # no e-commerce signal

    total = sum(scores.values())   # sum() over dict.values() gives the integer total; max = 12
    pct   = round(total / 12 * 100)  # convert to percentage for easy reading

    print(f"\n── Rubric: {label} ──")
    for crit, score in scores.items():
        bar = "█" * score + "░" * (3 - score)   # visual 3-block progress bar
        print(f"  {crit:<12} {bar}  {score}/3")
    print(f"  {'TOTAL':<12}        {total}/12  ({pct}%)")
    PASS_THRESHOLD = 10   # 83% pass threshold
    verdict = "PASS ✅" if total >= PASS_THRESHOLD else "FAIL ❌ — iterate"
    print(f"  Verdict: {verdict}")

    return {"scores": scores, "total": total, "pct": pct}

print("Rubric scorer defined.")

In [ ]:
# ── Section 7: Round 1 — evaluate the vague prompt ───────────────────────────

PROMPT_V1 = "Tell me about machine learning benefits."
# V1 is intentionally vague: no format, constraint, audience, or context

reply_v1 = ask(PROMPT_V1, label="improvement-v1", max_tokens=300)
print("V1 reply:")
print(reply_v1)

result_v1 = evaluate_response(reply_v1, label="V1 (vague)")
# result_v1 is a dict: {scores: {format:int, ...}, total:int, pct:int}
# We store it so we can compare with V2 in the next cell

In [ ]:
# ── Section 7: Round 2 — improve and re-evaluate ─────────────────────────────

PROMPT_V2 = (
    "You are explaining to a non-technical marketing manager at an e-commerce company."
    " List exactly 3 business benefits of machine learning for online stores."
    " Number each benefit 1, 2, 3."
    " Each benefit must be a single sentence of 20 words or fewer."
    " Avoid all technical jargon."
)
# V2 explicitly fills all four dimensions: audience, context, format, constraint

reply_v2 = ask(PROMPT_V2, label="improvement-v2", max_tokens=300)
print("V2 reply:")
print(reply_v2)

result_v2 = evaluate_response(reply_v2, label="V2 (improved)")

# ── Side-by-side comparison table ────────────────────────────────────────────
print("\n" + "=" * 60)
print("SIDE-BY-SIDE RUBRIC COMPARISON")
print("=" * 60)
header = f"{'Criterion':<14} {'V1 Score':>10} {'V2 Score':>10} {'Δ':>6}"
print(header)
print("-" * len(header))

criteria = list(result_v1["scores"].keys())   # ['format', 'constraint', 'audience', 'context']
for crit in criteria:
    v1_s = result_v1["scores"][crit]   # int score for this criterion in V1
    v2_s = result_v2["scores"][crit]   # int score for this criterion in V2
    delta = v2_s - v1_s                # positive = improvement; negative = regression
    delta_str = f"+{delta}" if delta > 0 else str(delta)   # format with explicit + sign for positive deltas
    print(f"  {crit:<12} {v1_s}/3 {'':>5} {v2_s}/3 {'':>5} {delta_str:>4}")

print("-" * len(header))
v1_total = result_v1['total']
v2_total = result_v2['total']
delta_total = v2_total - v1_total
delta_total_str = f"+{delta_total}" if delta_total > 0 else str(delta_total)
print(f"  {'TOTAL':<12} {v1_total}/12({'':1}{result_v1['pct']}%)  {v2_total}/12({result_v2['pct']}%)  {delta_total_str:>4}")
print("=" * 60)
print(f"  Improvement: {result_v1['pct']}% → {result_v2['pct']}%  (+{result_v2['pct'] - result_v1['pct']} percentage points)")

## Section 8: Failure Mode Analysis
## CCA objective demonstrated: Diagnose and fix common prompt failures with live evidence

| # | Failure Mode | Root Cause | Fix |
|---|-------------|------------|-----|
| 1 | Response too long / too short | No `max_tokens` set; no length constraint in prompt | Add `max_tokens` + word/sentence limit in prompt |
| 2 | Wrong format (prose instead of list) | Format not specified | State format explicitly: "numbered list", "JSON", "table" |
| 3 | Too technical / too simple | Audience not specified | State audience: "explain to a 10-year-old" or "assume PhD level" |
| 4 | Off-topic tangents | No context or scope constraint | Add domain/context: "focus only on X" |
| 5 | Inconsistent responses (same prompt, different output) | Prompt is ambiguous — multiple valid interpretations | Remove ambiguity; add examples of desired output |
| 6 | Truncated response | `max_tokens` too low | Raise `max_tokens`; check `stop_reason == 'max_tokens'` |

In [ ]:
# ── Section 8: Live demo — run a vague prompt 3× to expose inconsistency ─────

VAGUE = "Give me tips on productivity."   # maximally under-specified — no format, count, audience, context

print("Running the same vague prompt 3 times to measure output variance...\n")

run_a = ask(VAGUE, label="failure-demo-a", max_tokens=200)
run_b = ask(VAGUE, label="failure-demo-b", max_tokens=200)
run_c = ask(VAGUE, label="failure-demo-c", max_tokens=200)   # third run added to satisfy 3-sample standard

# ── Structural comparison ─────────────────────────────────────────────────────
def count_list_items(text):
    """Count lines that look like list items (numbered or bulleted).

    Uses re.findall with MULTILINE so ^ matches each line start, not just the
    beginning of the entire string.
    """
    # re.findall returns all non-overlapping matches as a list; len() gives the count
    # Pattern matches: '1.' '1)' '•' '-' '*' at the start of a line
    return len(re.findall(r'^\s*(?:\d+[.)\-]|[•\-\*])\s+\S', text, re.MULTILINE))

results = []
for i, run in enumerate([run_a, run_b, run_c], start=1):
    wc = len(run.split())          # word count: split() on whitespace gives a list of words
    li = count_list_items(run)     # list item count via regex
    results.append({"run": i, "word_count": wc, "list_items": li})
    print(f"Run {i}: {wc} words, {li} list items")
    print(f"  Preview: {run[:120]}...")
    print()

# ── Range analysis across all 3 runs ─────────────────────────────────────────
wc_values = [r["word_count"] for r in results]    # list comprehension: extract word_count from each dict
li_values = [r["list_items"] for r in results]    # same pattern for list item counts

print("=" * 50)
print("Variance across 3 runs of the same vague prompt:")
print(f"  Word-count range: min={min(wc_values)}, max={max(wc_values)}, range={max(wc_values)-min(wc_values)} words")
print(f"  List-item range:  min={min(li_values)}, max={max(li_values)}")
print()
print("▶ Large ranges reveal under-specification. A clear prompt collapses this variance.")

## Section 9: Token Usage Analysis
## CCA objective demonstrated: Track and interpret session-wide token spend including output variance across prompt versions

Token cost has two components:
- **Input tokens** — the prompt + conversation history sent to the API
- **Output tokens** — the tokens generated in the reply

Tighter prompts tend to produce more predictable, shorter replies — reducing output token spend.

In [ ]:
# ── Section 9: Token usage summary and output variance analysis ──────────────

# ── Full session log ──────────────────────────────────────────────────────────
print("FULL SESSION TOKEN LOG")
print(f"{'Label':<25} {'Input':>8} {'Output':>8} {'Total':>8} {'Truncated':>10}")
print("-" * 65)

total_input  = 0   # running totals; initialised to zero before the loop
total_output = 0

for entry in usage_log:   # iterate over every dict appended throughout the notebook
    total_in  = entry["input_tokens"] + entry["output_tokens"]   # combined token count for this call
    trunc_flag = "⚠️" if entry["truncated"] else ""               # visual flag for truncated responses
    print(f"  {entry['label']:<23} {entry['input_tokens']:>8} {entry['output_tokens']:>8} {total_in:>8} {trunc_flag:>10}")
    total_input  += entry["input_tokens"]   # accumulate session-wide input spend
    total_output += entry["output_tokens"]  # accumulate session-wide output spend

print("-" * 65)
print(f"  {'SESSION TOTAL':<23} {total_input:>8} {total_output:>8} {total_input+total_output:>8}")

# Cost estimate (Haiku pricing as of 2024; verify current pricing at anthropic.com/pricing)
COST_PER_1K_INPUT  = 0.00025   # USD per 1 000 input tokens
COST_PER_1K_OUTPUT = 0.00125   # USD per 1 000 output tokens
estimated_cost = (total_input / 1000 * COST_PER_1K_INPUT) + (total_output / 1000 * COST_PER_1K_OUTPUT)
print(f"\n  Estimated cost: ${estimated_cost:.5f} USD")
print("  (Pricing subject to change — always verify at anthropic.com/pricing)")

# ── Output token variance: V1 (vague) vs V2 (improved) ───────────────────────
print("\n" + "=" * 60)
print("OUTPUT TOKEN VARIANCE: VAGUE vs IMPROVED PROMPT")
print("=" * 60)

# Filter the usage_log for the two improvement-cycle calls
# dict lookup (entry['label']) is O(1) and exact; we use == for precise label matching
v1_entry = next((e for e in usage_log if e["label"] == "improvement-v1"), None)
v2_entry = next((e for e in usage_log if e["label"] == "improvement-v2"), None)

if v1_entry and v2_entry:
    v1_out = v1_entry["output_tokens"]   # output tokens for the vague prompt
    v2_out = v2_entry["output_tokens"]   # output tokens for the improved prompt
    delta  = v1_out - v2_out             # positive = improved prompt was more efficient

    print(f"  {'Version':<20} {'Output Tokens':>15}")
    print(f"  {'-'*35}")
    print(f"  {'V1 (vague)':<20} {v1_out:>15}")
    print(f"  {'V2 (improved)':<20} {v2_out:>15}")
    print(f"  {'-'*35}")
    direction = "reduced" if delta > 0 else "increased"
    print(f"  Delta:               {'-' if delta < 0 else '+'}{abs(delta):>14} tokens")
    print()
    print(f"  ▶ The improved prompt {direction} output tokens from {v1_out} to {v2_out}.")
    print("    A tighter prompt produces more predictable, cost-efficient responses.")
else:
    print("  (improvement-v1 or improvement-v2 not found in usage_log — re-run Section 7 cells)")

## Section 10: Practice Drills
## CCA objective demonstrated: Apply clarity principles independently to new prompts

Complete each drill by editing the `PROMPT` variable in the cell, then run the rubric scorer.

**Drill 1:** Rewrite the vague prompt `"Tell me about healthy eating"` to score ≥ 10/12 on the rubric below (format: numbered 3-item list; constraint: ≤15 words each; audience: busy parents; context: school lunches).  
**Drill 2:** Write a system prompt for a customer-support bot that handles only returns and refunds. Include all four system-prompt components (role, audience, format, constraint).  
**Drill 3:** Run your Drill 1 prompt 3 times. Print the word-count range. Is the range smaller than the vague version?

In [ ]:
# ── Drill 1: Rewrite a vague prompt ─────────────────────────────────────────
# TODO: Replace the vague prompt below with a clear, four-dimension version

DRILL1_PROMPT = "Tell me about healthy eating."   # ← edit this line

drill1_reply = ask(DRILL1_PROMPT, label="drill1", max_tokens=300)
print("Drill 1 reply:")
print(drill1_reply)

# Custom rubric for Drill 1
def rubric_drill1(text):
    """Score the Drill 1 response on format, constraint, audience, context."""
    scores = {}
    items = re.findall(r'^\s*\d+[.)]\.?\s+\S', text, re.MULTILINE)
    scores["format"]     = 3 if len(items) == 3 else (1 if len(items) in (2,4) else 0)
    long_lines           = [l for l in text.splitlines() if len(l.split()) > 15]
    scores["constraint"] = 3 if not long_lines else (1 if len(long_lines)==1 else 0)
    parent_words         = ["lunch", "kid", "child", "school", "parent", "meal", "pack"]
    found_parent         = any(re.search(r'\b'+w+r'\b', text.lower()) for w in parent_words)
    scores["audience"]   = 3 if found_parent else 0
    school_words         = ["lunch", "school", "lunchbox", "cafeteria", "snack"]
    found_school         = any(re.search(r'\b'+w+r'\b', text.lower()) for w in school_words)
    scores["context"]    = 3 if found_school else 0
    total = sum(scores.values())
    print("\nDrill 1 rubric:")
    for c, s in scores.items():
        print(f"  {c:<12} {s}/3")
    print(f"  TOTAL        {total}/12  — {'PASS ✅' if total >= 10 else 'FAIL ❌ — keep iterating'}")

rubric_drill1(drill1_reply)

In [ ]:
# ── Drill 2: Write a system prompt for a returns-and-refunds bot ─────────────
# TODO: Fill in all four system-prompt components

DRILL2_SYSTEM = (
    "You are a ..."           # ROLE: what is this bot?
    " Your audience is ..."   # AUDIENCE: who are the users?
    " Always respond by ..."  # FORMAT: how should replies be structured?
    " Only answer questions about ..."  # CONSTRAINT: what topics are in scope?
)

DRILL2_USER = "I bought a jacket three weeks ago and it arrived damaged. What are my options?"

drill2_reply = ask(DRILL2_USER, system=DRILL2_SYSTEM, label="drill2", max_tokens=200)
print("Drill 2 system prompt:")
print(DRILL2_SYSTEM)
print("\nReply:")
print(drill2_reply)

## Section 11: CCA Takeaways
## CCA objective demonstrated: Consolidate lesson concepts into exam-ready mental models

| # | Mental Model | One-Line Rule |
|---|-------------|---------------|
| 1 | **Four-dimension clarity** | Every prompt must specify format, constraint, audience, and context — unspecified dimensions are decided randomly by the model |
| 2 | **System = application layer** | The system prompt is evaluated before any user turn and is invisible to the user — it configures the model, not the conversation |
| 3 | **Stateless history accumulation** | The API has no memory; the full `messages` list is re-sent on every turn — input tokens grow linearly with conversation length |
| 4 | **Improve by rubric, not intuition** | Define a numeric rubric before writing the first prompt; use it to detect gaps and measure progress objectively |
| 5 | **Tighter prompt = cheaper output** | A well-constrained prompt produces shorter, more predictable replies — reducing output token spend and evaluation variance |

---

### Quick Reference Checklist
Before submitting any prompt to production:
- [ ] Format specified (list / JSON / prose / table)?
- [ ] Length / count constraint stated?
- [ ] Audience named explicitly?
- [ ] Domain / context scoped?
- [ ] `system` used for persistent rules; user turn used for per-request content?
- [ ] `max_tokens` set and `stop_reason` checked?